# NAB Anomaly Detection Baseline

This notebook inspects the saved Phase 3 experiment artifacts for the NAB anomaly-detection baseline. It does not retrain models or duplicate experiment logic. Reproduce the artifacts with:

```powershell
.\.venv\Scripts\python.exe scripts\experiments\run_nab_baseline.py --run-id run_phase3_baseline
```

The script remains the source of truth for training, thresholding, metrics, predictions, and plots.

In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import pandas as pd
from IPython.display import Image, Markdown, display

project_root = Path.cwd().resolve()
if project_root.name == "notebooks":
    project_root = project_root.parent

run_dir = (
    project_root / "experiments" / "anomaly" / "nab" / "isolation_forest" / "run_phase3_baseline"
)
if not run_dir.exists():
    candidates = sorted(
        (project_root / "experiments" / "anomaly" / "nab" / "isolation_forest").glob("run_*")
    )
    if not candidates:
        raise FileNotFoundError(
            "No NAB baseline run found. Run scripts/experiments/run_nab_baseline.py first."
        )
    run_dir = candidates[-1]

run_dir

In [ ]:
with (run_dir / "metrics.json").open("r", encoding="utf-8") as file:
    metrics = json.load(file)

with (run_dir / "dataset_summary.json").open("r", encoding="utf-8") as file:
    dataset_summary = json.load(file)

per_series = pd.read_csv(run_dir / "per_series_metrics.csv")

display(Markdown(f"## Run: `{run_dir.name}`"))
display(pd.DataFrame([dataset_summary]).T.rename(columns={0: "value"}))

## Global Metrics

These are measured on the test split only. Labels are not used for training or threshold selection.

In [ ]:
global_rows = []
for model_name, values in metrics["global"].items():
    delay = values["detection_delay"]
    global_rows.append(
        {
            "model": model_name,
            "precision": values["precision"],
            "recall": values["recall"],
            "f1": values["f1"],
            "pr_auc": values["pr_auc"],
            "false_positive_rate": values["false_positive_rate"],
            "detected_windows": delay["detected_windows"],
            "missed_windows": delay["missed_windows"],
            "median_delay_steps": delay["median_delay_steps"],
        }
    )

global_metrics = pd.DataFrame(global_rows)
display(global_metrics)

## Per-Series Distribution

The median F1 is important here: a global improvement can still hide many failed individual series.

In [ ]:
distribution_rows = []
for model_name, field_values in metrics["per_series_distribution"].items():
    row = {"model": model_name}
    for field_name, summary in field_values.items():
        row[f"{field_name}_mean"] = summary["mean"]
        row[f"{field_name}_median"] = summary["median"]
    distribution_rows.append(row)

display(pd.DataFrame(distribution_rows))

In [ ]:
for model_name in sorted(per_series["model"].unique()):
    display(Markdown(f"### Best series by F1: `{model_name}`"))
    cols = ["entity_id", "precision", "recall", "f1", "pr_auc", "positives", "threshold"]
    display(
        per_series.loc[(per_series["model"] == model_name) & (per_series["positives"] > 0), cols]
        .sort_values(["f1", "recall", "precision"], ascending=False)
        .head(10)
    )

## Saved Figures

The figures below are generated by the experiment script and saved as run artifacts.

In [ ]:
for figure_path in sorted((run_dir / "figures").glob("*.png")):
    display(Markdown(f"### `{figure_path.name}`"))
    display(Image(filename=str(figure_path)))

## Error Analysis

Review missed windows, delayed detections, and false-positive examples before tuning thresholds or adding more complex models.

In [ ]:
with (run_dir / "error_analysis.json").open("r", encoding="utf-8") as file:
    error_analysis = json.load(file)

for model_name, sections in error_analysis.items():
    display(Markdown(f"### `{model_name}`"))
    for section_name, rows in sections.items():
        display(Markdown(f"#### {section_name}"))
        display(pd.DataFrame(rows).head(10))

## Notes

- Isolation Forest improves global precision, recall, F1, and PR-AUC over rolling z-score in this run.
- The false positive rate is higher for Isolation Forest.
- Median per-series F1 remains zero for both approaches, so Phase 4 should not assume this baseline is sufficient.
- This notebook intentionally reports measured artifacts only; update it by rerunning the experiment script, not by manually editing results.